# 1. Utility functions
### Main modifications

#### Activation function

The activation function selector was updated in several ways:

- `F.tanh` was replaced by `torch.tanh`.
- `F.sigmoid` was replaced by `torch.sigmoid`.

These changes avoid the use of deprecated PyTorch functions while preserving exactly the same mathematical behaviour.

Additionally, a validation step was introduced:

```python
if act_fun not in act_dict:
    raise ValueError(...)
```

This produces a clear error message whenever an unsupported activation function is specified instead of generating an obscure exception later during training.

#### Optimizer selection

The optimizer function underwent several important modifications.

The original implementation only supported:

- AdamW
- L-BFGS

Furthermore, invalid optimizer names automatically fell back to AdamW.

The modified implementation introduces:

- Adam
- AdamW
- L-BFGS

and replaces the silent fallback with

```python
raise ValueError(...)
```

This avoids accidentally training the network with an unintended optimizer.
#### Scheduler

The scheduler logic remains essentially unchanged.

However, unnecessary recursive calls were removed and invalid scheduler names now generate explicit exceptions, making configuration errors easier to identify.

# 2. Fourier feature layers
Only minor modifications were introduced in the Fourier feature implementation.

The most important changes are:

- the random projection matrix is now registered as a buffer

```python
self.register_buffer(...)
```

instead of being stored as a regular tensor.

This ensures that:

- it is automatically transferred to GPU,
- it is saved together with the model,
- it is not treated as a trainable parameter.

Additionally, input shape validation was added to ensure that the coordinate tensor has shape `(N,1)`.

This prevents dimension mismatch errors before the forward computation.

# 3. Adaptive Linear layer
The AdaptiveLinear class preserves the original adaptive activation mechanism proposed by Jagtap et al.

Only one implementation detail was modified.

Originally, the code checked

```python
if self.adaptive_rate:
```

The modified implementation uses

```python
if self.adaptive_rate is not None:
```

This avoids ambiguities when the adaptive rate equals zero while maintaining identical behaviour in all other cases.

# 4. Loss functions

In [ ]:
# Original

L2relLoss
L2relLossMultidim
MSE
H1relLoss_fourier

# Modified

L2relLoss
MSE
H1relLoss_fourier

### Removed losses

The original repository included an additional multidimensional relative L2 loss (`L2relLossMultidim`).

Since the present project predicts only a single voltage trajectory instead of multiple coupled variables, this loss function was removed.

Its removal simplifies the code without affecting the proposed methodology.

### L2 Relative Loss

The computation changed from

```python
torch.sum(...)
```

to

```python
torch.mean(...)
```

As a consequence, the loss becomes independent of the batch size, making training behaviour more consistent across different mini-batch configurations.

### Mean Squared Error

Similarly, the MSE loss now returns

```python
torch.mean(...)
```

instead of

```python
torch.sum(...)
```

This follows the standard PyTorch convention for regression losses.

# 5. Removed neural network architectures
One of the major simplifications concerns the removal of several architectures that are not required for this dissertation.

The following components were removed:

- MLP
- ResidualBlockCNN
- ResNet
- TimeDistributed
- myGRU

These architectures were originally included to support alternative operator-learning experiments.

Since the proposed framework relies exclusively on DeepONet, they unnecessarily increased the complexity of the repository and were therefore removed.
Removing these classes considerably reduces:

- file length;
- maintenance effort;
- compilation time;
- cognitive complexity for future users.

# 6. Feed-forward neural networks
The three fully-connected architectures

- FNN
- FNN_BN
- FNN_LN

remain essentially unchanged.

Only readability improvements and additional comments were introduced.

Consequently, the numerical behaviour of the DeepONet architecture is preserved.